#PySpark Practice

In [0]:
print ("hello world")

#Create DataFrame using Python Objects

In [0]:
# Create a DataFrame from a list of tuples.
#First argument is an array of tuples (Rows) and second is an array of column names 
# Create a DataFrame with column names specified.
spark.createDataFrame([('Alice', 1)], ['name', 'age']).show()
# +-----+---+
# | name|age|
# +-----+---+
# |Alice|  1|
# +-----+---+

df = spark.createDataFrame([('Alice', 14, False), ('Sugar', 13, True)], ['name', 'age'])
#+-----+---+-----+
#| name|age|   _3|
#+-----+---+-----+
#|Alice| 14|false|
#|Sugar| 13| true|
#+-----+---+-----+

#   Notes : This gives the Column Name from the second array in the arguments.
#	        If we miss a column name in the second array argument then the column will _# # is the column number 
#			Also determines the datatype, but using the first value. i.e. if we give 1 and 1.2 it will fail [CANNOT_MERGE_TYPE] Can not merge type `DoubleType` and `LongType`.


#below shows the data in table format 
#df.display()

##
print(df)  #returns the below schema list
#DataFrame[name: string, age: bigint, _3: boolean]

In [0]:
# Create a DataFrame from a list of dictionaries.
spark.createDataFrame([{'name': 'Alice', 'age': 10}, {'name': 'Bob', 'age': 20}])
df.display()
# +---+-----+
# |age| name|
# +---+-----+
# |  1|Alice|
# +---+-----+
#    Notes : This gives the Column Name from the struct 
#  			 Also determines the datatype, but using the first value. i.e. if we give 1 and 1.2 it will fail [CANNOT_MERGE_TYPE] Can not merge type `DoubleType` and `LongType`.


In [0]:
# Create a DataFrame with an explicit schema. Note : We will have to import the "types" shown below 
#https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/data_types.html

from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BooleanType, CharType
schema = StructType([
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("pass", BooleanType(), True)])
df = spark.createDataFrame([('Alice', 1.0, True), ('Sugar',2, False)], schema)
display (df)
#+-----+---+-----+
#| name|age| pass|
#+-----+---+-----+
#|Alice|  1| true|
#|Sugar|  2|false|
#+-----+---+-----+

#   Notes : This gives the Column Name from the schema variable which is created using StructType / StructField.
#			Also casts the Float to IntegerType automatically 
#			To insert a Date use the below 

In [0]:
#Note : createDataFrame cannot infer datatype Date
#For DateType it is not possible to pas a string. We will have to use the Python Datetime module
import datetime as Datetime 
from pyspark.sql.types import StructType, StructField, DateType
schema = StructType([
    StructField("name", DateType(), True)])
  
df = spark.createDataFrame([(Datetime.date(2025,12,12))], schema)
display (df) 

#df = spark.createDataFrame([("2025-12-12")], schema)  #This will fail with below error message 
#ArrowTypeError: object of type <class 'str'> cannot be converted to int
#wsfs/fuse/wsfs.go:964 Error(message=Cannot find child LearnDatabricks, errno=no such file or directory, statusCode=0, stack=<nil>)
#[Trace ID: 00-e88ea90823c9249909a0617f56685e2a-4c51af4c15b29013-00]

In [0]:
# Create a DataFrame with a DDL-formatted schema string.
spark.createDataFrame([('Alice', 1)], "name: string, age: int").show()
# +-----+---+
# | name|age|
# +-----+---+
# |Alice|  1|
# +-----+---+


In [0]:
# Create a DataFrame from Row objects.
from pyspark.sql import Row
Person = Row('name', 'age')
spark.createDataFrame([Person("Alice", 1)]).show()
# +-----+---+
# | name|age|
# +-----+---+
# |Alice|  1|
# +-----+---+		

In [0]:
%python
################################################################################################
#                          Sample dataframe for trying out commands                            #
################################################################################################
import pyspark.sql.types as T

emp_data = [
    [1,101,"John Doe",30,"Male",50000,"2015-01-01"],
    [2,101,"Jane Smith",25,"Female",62000,"2016-02-15"],
    [3,102,"Bob Brown",35,"Male",55000,"2014-05-01"],
    [4,102,"Alice Lee",28,"Female",49000,"2017-09-30"]
]

emp_schema = T.StructType([
    T.StructField("id", T.IntegerType(), True),
    T.StructField("dept_id", T.IntegerType(), True),
    T.StructField("name", T.StringType(), True),
    T.StructField("age", T.IntegerType(), True),
    T.StructField("gender", T.StringType(), True),
    T.StructField("salary", T.DoubleType(), True),
    T.StructField("start_date", T.StringType(), True)  #Note this is StringType and not DateType. If it is DateType we will have to use the Python Datetime module
])

emp = spark.createDataFrame(emp_data, emp_schema)

In [0]:
#to Display in tabular format 
emp.display()

In [0]:
#to show the output in text but tabular format 
emp.show()

In [0]:
#below give the number of rows in the dataframe
emp.count()

#aggregate 
emp.agg({"*": "count"}).show()
#+--------+
#|count(1)|
#+--------+
#|       4|
#+--------+

emp.agg({"salary": "sum"}).show()
#+-----------+
#|sum(salary)|
#+-----------+
#|   216000.0|
#+-----------+

emp.agg({"age": "max"}).show()
#+--------+
#|max(age)|
#+--------+
#|      35|
#+--------+

#for multiple columns
emp.agg({"age": "max", "salary":"sum"}).show()
#+--------+-----------+
#|max(age)|sum(salary)|
#+--------+-----------+
#|      35|   216000.0|
#+--------+-----------+

#Note cannot get multiple agg in the same line. Does not fail but it shows the last one only
emp.agg({"age": "max", "age": "min"}).show()

#To get multiple agg in the same line, use the below whe eill have to use the functions 


In [0]:
#https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html
import pyspark.sql.functions as F

emp.select(F.max("age"), F.min("age")).show()

In [0]:
#This is like a pointer to the dataframe and can be used when we do self join 
emp_side = emp.alias("emp")
mgr_side = emp.alias("mgr")

In [0]:
#DataFrame.checkpoint() is a powerful mechanism used to truncate the lineage (execution plan) of a DataFrame and save its current state to a reliable, persistent storage system (like HDFS, S3, or ADLS).
#You should incorporate df.checkpoint() into your data pipelines when:
#   You are building loops: You are updating a DataFrame inside a for or while loop repeatedly (common in graph processing, machine learning, or processing multi-level corporate hierarchies).
#   You are encountering StackOverflowError: Your Spark driver is crashing during plan optimization phases due to an unmanageably large DAG.
#   Resilience across jobs is required: You want to save an intermediate state of an incredibly expensive calculation so that if a long-running multi-hour script crashes in Phase 2, you don't have to restart Phase 1 from scratch.



In [0]:
#### DataFrame.cache() should be used when you intend to reuse the exact same DataFrame multiple times in downstream actions within your Spark application.

## 1. Read and perform expensive cleaning
#df_clean = spark.read.parquet("raw_data/").filter(...).join(...)

## 2. Cache it because it's the foundation for multiple downstream steps
#df_clean.cache()

## Action A: Write a summary report
#df_clean.groupBy("Country").count().write.mode("overwrite").parquet("reports/summary/")

## Action B: Calculate top-tier metrics
#total_revenue = df_clean.agg({"Revenue": "sum"}).collect()[0][0]

## Action C: Save specific records
#df_clean.filter(F.col("Status") == "Flagged").write.jdbc(...)

#Without cache(): Spark will execute the read, filter, and join steps three separate times (once for Action A, once for Action B, and once for Action C).
#With cache(): Spark executes the cleaning logic once for Action A, saves it in RAM, and Action B and C pull directly from memory in a fraction of a second.

#unpersist() to Free Up Memory (Memory is the most precious resource in a Spark cluster. 
#If you cache a DataFrame in Phase 1 of your script, and you no longer need it in Phase 4, you should explicitly delete it from RAM to prevent cluster choking:
#df.unpersist() 


In [0]:
import pyspark.sql.functions as F

#.collect() is to get data a row/column or rows into python list
#This is not recommended as it will get all the data into the driver node and can crash the driver node if the data is too big

#forExample if we want the unique dept_id into a list of rows for looping each dept
depts = emp.select("dept_id").distinct().collect()
print(depts)

type(depts)

for row in depts:
    print(row["dept_id"])

#example 2
summary=emp.agg(F.count("*").alias("count"),F.sum("salary").alias("totalSalary")).collect()
print(summary)

print(f"Number of employees {summary[0]["count"]}, Total Salary {summary[0]["totalSalary"]}")

In [0]:
from pyspark.sql import functions as F

cols = emp.columns
type(cols)
print(cols)

for col in cols:
    print(col)

#Imagine you have a data lake table with 200 columns. Regulations mandate that you must drop or mask any column containing sensitive Personally Identifiable Information (PII) before saving it to a public directory, 
# or drop metadata columns created by an ingestion tool (like columns starting with _sys_).
#Instead of typing out all those columns to drop them, you use df.columns to dynamically find them



# Fetch all columns dynamically
all_cols = emp.columns

# Use Python logic to find any column containing "ssn", "password", or "sys"
pii_and_sys_cols = [
    c for c in all_cols 
    if "name" in c.lower() or "salary" in c.lower() or c.startswith("_sys_")
]

# Drop them all dynamically in one pass
df_secure = emp.drop(*pii_and_sys_cols)

print(df_secure.columns)

#or maybe we want to uppercase all coulmns 
all_cols = emp.columns

upper_cols = [c.upper() for c in all_cols]
    
df_upper = emp.toDF(*upper_cols)  #The .toDF() method is a quick, versatile utility used to rename columns or apply a new schema to an existing DataFrame.
print(df_upper.columns)


In [0]:
#Computes basic statistics for numeric and string columns.
#This includes count, mean, stddev, min, and max. If no columns are given, this function computes statistics for all numerical or string columns.

emp.describe().display()

In [0]:
#Removes distinct rows from the DataFrame.

from pyspark.sql import functions as F
import pyspark.sql.types as T

emp_data = [
    [1,"Alice Lee","Male"],
    [2,"Jane Smith","Female"],
    [3,"Bob Brown","Male"],
    [1,"Alice Lee","Male"],
    [3,"Bob Brown","Male"],
    [2,"Jane Smith","Male"]
]

emp_schema = T.StructType([
    T.StructField("id", T.IntegerType(), True),
    T.StructField("name", T.StringType(), True),
    T.StructField("gender", T.StringType(), True)
])

emp = spark.createDataFrame(emp_data, emp_schema)

emp.sort("id").display()
emp.distinct().sort("id").display()

#if we want disctinct values of a column then select before distinct 
emp.select("name").distinct().display()

#if we want disctinct values for multiple columns then select before distinct 
emp.select("name", "gender").distinct().show()

In [0]:
#drop column from a dataframe 

emp.drop("gender").display()

emp.drop("gender","id").display()

a=["gender", "name"]
emp.drop(*a).display()


In [0]:
#the same result as distinct(), distinct() is a synonym for dropDuplicates() without parameter columns 
emp.dropDuplicates().display()

#using arguments. This is useful when you want to drop duplicates based on a subset of columns
#When you pass a subset of columns into dropDuplicates(), Spark groups the rows by those specific columns. For any column not included in that list, Spark applies a "First One Wins" rule.
#The value selected will come from the very first row that Spark's engine physically processes for that group. Because Spark is a distributed computing system, this selection is non-deterministic (unpredictable) by default.
emp.dropDuplicates(["id","name"]).display()

##you can see that whin I use the orderby before dropDuplicates, the result is different
emp.orderBy("gender", ascending=False).dropDuplicates(["id","name"]).display()

In [0]:
from pyspark.sql import Row
df = spark.createDataFrame([
    Row(age=10, height=80.0, name="Alice"),
    Row(age=5, height=float("nan"), name="Bob"),
    Row(age=None, height=None, name="Tom"),
    Row(age=None, height=float("nan"), name=None),
])

#both the below give same rusult. drops rows which have null values in atleast one column
df.na.drop(how="any").display()
df.dropna(how="any").display()


##only if all columns have null values, then only it will drop the row
df.na.drop(how="all").display()
df.dropna(how="all").display()

#threshold means atleast how many columns should have null values to drop the row 
#thresh defines an exact integer requirement: "Keep only the rows that have at least $n$ non-null values."  
#argument how is ignore when we use thresh
df.na.drop(how="any", thresh=2).display()
df.dropna(how="any", thresh=2).display()



In [0]:

#Returns all column names and their data types as a list.
print(emp.dtypes)

In [0]:

df1 = spark.createDataFrame([("a", 1), ("a", 1), ("a", 1), ("a", 2), ("b",  3), ("c", 4)], ["C1", "C2"])
df2 = spark.createDataFrame([("a", 1), ("b", 3)], ["C1", "C2"])   #, ("a", 1), ("a", 1)

#Note : It only removes the first occurance of ("a",1) from df1, if you want the other then repeat the entry in df2
df1.exceptAll(df2).show()

In [0]:
df1 = spark.createDataFrame([("a", 1), ("a", 1), ("a", 1), ("a", 2), ("b",  3), ("c", 4)], ["C1", "C2"])
df2 = spark.createDataFrame([("a", 1), ("b", 3)], ["C1", "C2"])   #, ("a", 1), ("a", 1)

#Note : this will remove all occurance of ("a",1)
df1.subtract(df2).show()

In [0]:
#The exists method provides a way to create a boolean column that checks for the presence of related records in a subquery. When applied within a DataFrame, this method allows you to filter rows based on whether matching records exist in the related dataset. The resulting Column object can be used directly in filtering conditions or as a computed column.

from pyspark.sql import functions as sf
df1 = spark.createDataFrame([("a", 1), ("x", 7), ("y", 9), ("a", 2), ("b",  3), ("c", 4)], ["C1", "C2"])
df2 = spark.createDataFrame([("a", 1), ("b", 3)], ["C1", "C2"])   #, ("a", 1), ("a", 1)

df1.alias("a").where(df2.alias("b").where(sf.col("b.C1") == sf.col("a.C1").outer()).exists()).orderBy("C1").show()

#When you write a nested subquery (a query inside another query), Spark isolates the inner query's execution context. 
#If the inner query needs to look "up" and reference a column from the main table outside itself, you must explicitly tag that column with .outer().
#Without .outer(), Spark's analyzer will only look inside the immediate subquery, fail to find or correctly bind the outer column, and throw an error.

In [0]:
df = spark.createDataFrame(
    [(14, "Tom"), (23, "Alice"), (16, "Bob")], ["age", "name"])
df.explain()  

df.explain(extended=True)

df.explain(mode="formatted")

#df.explain(extended=True, mode="formatted")     fails with below error 
#[CANNOT_SET_TOGETHER] extended and mode should not be set together.



In [0]:
#based on the datatype of the value provided it populates that value into all columns of that datatype in dataFrame 
df = spark.createDataFrame([
    (10, 80.5, "Alice", "Lee", None),
    (5, None, "Bob", "Smith", False),
    (None, None, "Tom", None, None),
    (None, None, None, None, True)],
    schema=["age", "height", "name", "surname", "bool"])
df.printSchema()

#as "N/A" is string, it will replace NULLS in columns name and surname 
df.fillna("N/A").show()

#this will fill both age and height 
df.fillna(9999).show()

#this will fill both name and surname as the numer is in ""
df.fillna("9999").show()

#this will fill the bool column
df.fillna(True).show()

#although I am specifying subset, it will fill not fill any as the value is in ""
df.fillna("9999", subset="age").show

#although I am specifying subset, it will fill not fill any as the value is in ""
df.fillna(99, subset="age").show

